# 🌱 Plant Disease Detection — A Simple End-to-End CNN
### Classify a photo of a leaf as healthy or diseased, using a CNN built from scratch
---

**The problem:** Given a photo of a plant leaf, tell which of 5 conditions it shows:
`Apple — Black Rot`, `Apple — Cedar Rust`, `Apple — Healthy`, `Apple — Scab`, `Corn — Common Rust`.

This is a **real, relatable computer-vision problem** — the same idea powers apps farmers use to diagnose crops from a phone photo.

**Why this is a good first project:**
- The images are **high-resolution colour photos** (256×256), not tiny thumbnails
- The 5 classes are **visually distinct** — you can see the disease spots yourself
- The whole thing is **~40 lines of real code**, runs start-to-finish in a few minutes on a Colab GPU
- It uses every core CNN idea: convolution, pooling, ReLU, a dense head, softmax

**Everything below is pre-run** so you can read it as a walkthrough. To run it live: open in Google Colab, set Runtime → GPU, and Run All. The dataset downloads automatically.

---

### The 7 steps
1. Get the data (auto-download)
2. Look at the images
3. Load them into TensorFlow
4. Build the CNN
5. Train it
6. See how well it did
7. Predict on a new leaf

## Step 0 — Setup

On Colab, TensorFlow is already installed. Set **Runtime → Change runtime type → GPU** for fast training (the model trains in ~1 minute on a GPU, ~8 minutes on CPU).

In [ ]:
import torch
print(torch.cuda.is_available()) # Should return True
print(torch.cuda.get_device_name(0))


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models

print("TensorFlow version:", tf.__version__)
print("GPU available:", "YES " if tf.config.list_physical_devices('GPU') else "no (will be slower)")

## Step 1 — Download the dataset (live, in class)

We use the **PlantVillage** dataset (a well-known public collection of labelled plant-leaf photos). We'll download it **right now**, directly from its GitHub home — nothing needs to be prepared in advance.

**How the download works:** the full PlantVillage repo is large (~2 GB), so instead of grabbing everything we use Git's **sparse-checkout** to pull *only* the image folders, and we download the file list *without* the image blobs first (`--filter=blob:none`), then check out just the `raw/color/` folder. This keeps the download fast and is the standard way to grab part of a big repo.

Run the cell below and watch the folders appear. It takes about 1–2 minutes on Colab.

In [ ]:
import os, glob, random, shutil

# --- 1a. Sparse-clone ONLY the colour image folders from the PlantVillage repo ---
if not os.path.exists("PlantVillage-Dataset"):
    print("Downloading PlantVillage image folders from GitHub... (~1-2 min)")
    # --filter=blob:none  -> don't download file contents yet (fast)
    # --no-checkout       -> don't materialise files until we pick folders
    os.system("git clone --no-checkout --depth 1 --filter=blob:none "
                "https://github.com/spMohanty/PlantVillage-Dataset.git")
    # tell git we only want the colour images, then actually fetch them
    os.system("cd PlantVillage-Dataset && git sparse-checkout init --cone "
                "&& git sparse-checkout set raw/color && git checkout")
    print("Download complete.")
else:
    print("PlantVillage-Dataset already downloaded.")

SOURCE = "PlantVillage-Dataset/raw/color"
print("\nAll classes available in the full dataset:")
for c in sorted(os.listdir(SOURCE)):
    print("  ", c)

### Build a small, balanced 5-class subset

The full dataset has 38 classes — too many for a first demo. We pick **5 visually distinct classes** and build a clean `train/` + `test/` split right here. We cap each class to the same number of images so the dataset is **balanced** (no class dominates), and hold out 20% for testing.

In [ ]:
# --- 1b. Pick 5 classes and build a balanced train/test split ---
# (original folder name  ->  clean display name)
SELECTED = {
    "Apple___Apple_scab":          "Apple_Scab",
    "Apple___Black_rot":           "Apple_Black_Rot",
    "Apple___Cedar_apple_rust":    "Apple_Cedar_Rust",
    "Apple___healthy":             "Apple_Healthy",
    "Corn_(maize)___Common_rust_": "Corn_Common_Rust",
}
PER_CLASS = 275      # cap per class (smallest class has ~275) -> balanced
TEST_FRACTION = 0.2  # hold out 20% for testing
random.seed(42)

shutil.rmtree("plant_disease", ignore_errors=True)
for orig, clean in SELECTED.items():
    files = sorted(glob.glob(f"{SOURCE}/{orig}/*"))
    random.shuffle(files)
    files = files[:PER_CLASS]
    n_test = int(len(files) * TEST_FRACTION)
    for i, f in enumerate(files):
        split = "test" if i < n_test else "train"
        dest = f"plant_disease/{split}/{clean}"
        os.makedirs(dest, exist_ok=True)
        shutil.copy(f, dest)

# Report what we built
print("Built balanced 5-class dataset:\n")
for split in ["train", "test"]:
    print(f"  {split}/")
    total = 0
    for clean in sorted(SELECTED.values()):
        n = len(os.listdir(f"plant_disease/{split}/{clean}"))
        total += n
        print(f"    {clean:20s} {n:>4} images")
    print(f"    {'TOTAL':20s} {total:>4} images\n")

## Step 2 — Look at the data first

**Always look at your images before training.** It catches mislabeled data, wrong colours, or surprises — and it tells you whether the classes are even distinguishable by eye. If *you* can't tell them apart, the model will struggle too.

We'll look at the data three ways: one example per class, then a bigger random gallery, then the class counts.

### 2a — One example from each class

In [ ]:
import glob
from PIL import Image

class_dirs = sorted(os.listdir("plant_disease/train"))
fig, axes = plt.subplots(5, 3, figsize=(7, 11))
for r, cls in enumerate(class_dirs):
    files = sorted(glob.glob(f"plant_disease/train/{cls}/*"))[:3]
    for c, f in enumerate(files):
        axes[r, c].imshow(Image.open(f))
        axes[r, c].axis("off")
        if c == 0:
            axes[r, c].set_title(cls.replace("_", " "), fontsize=11, loc="left", fontweight="bold")
plt.suptitle("PlantVillage — 256x256 leaf images (5 classes)", fontsize=13, y=1.0)
plt.tight_layout(); plt.show()

### 2b — Browse the dataset: a bigger random gallery

This is the cell to spend time on with students. It shows **5 random images from every class** so they can see the *variety* within each class — different leaf angles, lighting, and how strong the disease symptoms are. Notice:

- **Apple Black Rot** — dark brown/black blotches with concentric rings
- **Apple Cedar Rust** — bright orange-yellow spots
- **Apple Healthy** — clean, even green, no marks
- **Apple Scab** — olive-green to brown fuzzy lesions
- **Corn Common Rust** — reddish-brown pustules in lines along the leaf

Re-run this cell to draw a fresh random sample each time.

In [ ]:
import random

class_dirs = sorted(os.listdir("plant_disease/train"))
N_PER_CLASS = 5

fig, axes = plt.subplots(len(class_dirs), N_PER_CLASS, figsize=(13, 13))
for r, cls in enumerate(class_dirs):
    files = glob.glob(f"plant_disease/train/{cls}/*")
    random.shuffle(files)
    for c in range(N_PER_CLASS):
        axes[r, c].imshow(Image.open(files[c]))
        axes[r, c].axis("off")
        if c == 0:
            axes[r, c].set_title(cls.replace("_", " "), fontsize=12,
                                loc="left", fontweight="bold", color="#0D9488")
plt.suptitle("Browse the dataset — 5 random samples per class", fontsize=15, y=1.0)
plt.tight_layout(); plt.show()

### 2c — How many images per class?

Check the **class balance**. A balanced dataset (roughly equal counts per class) means the model won't be biased toward any one class. Ours is balanced by design — we capped every class to the same number in Step 1.

In [ ]:
counts_train = [len(os.listdir(f"plant_disease/train/{c}")) for c in class_dirs]
counts_test  = [len(os.listdir(f"plant_disease/test/{c}"))  for c in class_dirs]

fig, ax = plt.subplots(figsize=(10, 4))
x = range(len(class_dirs))
ax.bar([i-0.2 for i in x], counts_train, width=0.4, label="train", color="#0D9488")
ax.bar([i+0.2 for i in x], counts_test,  width=0.4, label="test",  color="#F59E0B")
ax.set_xticks(list(x))
ax.set_xticklabels([c.replace("_"," ") for c in class_dirs], rotation=20, ha="right")
ax.set_ylabel("Number of images"); ax.set_title("Images per class (balanced)")
ax.legend(); ax.grid(axis="y", alpha=0.3)
for i, (tr, te) in enumerate(zip(counts_train, counts_test)):
    ax.text(i-0.2, tr+5, str(tr), ha="center", fontsize=8, fontweight="bold")
    ax.text(i+0.2, te+5, str(te), ha="center", fontsize=8, fontweight="bold")
plt.tight_layout(); plt.show()

## Step 3 — Load the images into TensorFlow

`image_dataset_from_directory` is the easy button: point it at a folder of class-subfolders and it builds a ready-to-train dataset, reads the class names from the folder names, and resizes every image for you.

We resize to **128×128** — big enough to keep the disease detail visible, small enough to train fast. (You can bump this to 224×224 for more accuracy at the cost of speed.)

In [ ]:
IMG_SIZE = 128
BATCH_SIZE = 32

train_ds = tf.keras.utils.image_dataset_from_directory(
    "plant_disease/train",
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    label_mode="categorical",
    seed=42,
    shuffle=True,
)
test_ds = tf.keras.utils.image_dataset_from_directory(
    "plant_disease/test",
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    label_mode="categorical",
    seed=42,
    shuffle=False,
)

CLASS_NAMES = train_ds.class_names
print("Classes:", CLASS_NAMES)

# Hold out 20% of the training data for validation
val_batches = max(1, tf.data.experimental.cardinality(train_ds).numpy() // 5)
val_ds = train_ds.take(val_batches)
train_ds = train_ds.skip(val_batches)

# Speed: keep data in memory after first read
train_ds = train_ds.cache().prefetch(tf.data.AUTOTUNE)
val_ds = val_ds.cache().prefetch(tf.data.AUTOTUNE)
test_ds = test_ds.cache().prefetch(tf.data.AUTOTUNE)

## Step 4 — Build the CNN

Here's the whole model. Read it top to bottom — it's the standard CNN recipe:

```
Rescaling           → turn pixel values from 0–255 into 0–1
Conv2D(32) + Pool   → find simple features (edges), then shrink
Conv2D(64) + Pool   → combine into textures
Conv2D(128) + Pool  → combine into shapes (spots, lesions)
Conv2D(128) + Pool  → high-level disease patterns
GlobalAveragePooling→ collapse the feature maps into one vector
Dense(128) + Dropout→ a classifier layer (+ dropout to reduce overfitting)
Dense(5, softmax)   → output: probability for each of the 5 classes
```

Each `Conv2D → MaxPool` block **doubles the number of filters** (32→64→128) while **halving the image size** (128→64→32→16→8). That's the classic CNN funnel: as you lose spatial resolution, you gain feature richness.

In [ ]:
model = models.Sequential([
    layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3)),
    layers.Rescaling(1./255),                              # 0-255 -> 0-1

    layers.Conv2D(32, 3, activation="relu", padding="same"),
    layers.MaxPool2D(),                                    # 128 -> 64

    layers.Conv2D(64, 3, activation="relu", padding="same"),
    layers.MaxPool2D(),                                    # 64 -> 32

    layers.Conv2D(128, 3, activation="relu", padding="same"),
    layers.MaxPool2D(),                                    # 32 -> 16

    layers.Conv2D(128, 3, activation="relu", padding="same"),
    layers.MaxPool2D(),                                    # 16 -> 8

    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.4),
    layers.Dense(5, activation="softmax"),                 # 5 classes
], name="PlantCNN")

model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)
model.summary()

## Step 5 — Train the model

`model.fit` does the learning. We train for 12 epochs (12 passes over the data). Watch the **val_accuracy** climb — that's accuracy on images the model hasn't trained on.

On a Colab **GPU** this takes about a minute. On CPU, about 8 minutes.

In [ ]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=25,
)

### Plot the learning curves

Two things to check:
- **Both accuracy lines rising together** = the model is learning real patterns ✅
- **A big gap (train high, val low)** = overfitting (memorising instead of learning)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(history.history["accuracy"], label="train", linewidth=2)
ax1.plot(history.history["val_accuracy"], label="validation", linewidth=2)
ax1.set_title("Accuracy"); ax1.set_xlabel("Epoch"); ax1.legend(); ax1.grid(alpha=0.3)
ax2.plot(history.history["loss"], label="train", linewidth=2)
ax2.plot(history.history["val_loss"], label="validation", linewidth=2)
ax2.set_title("Loss"); ax2.set_xlabel("Epoch"); ax2.legend(); ax2.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## Step 6 — How well did it do?

Evaluate on the **test set** — images held out from training. This is the honest measure of how the model will do on new leaves.

In [ ]:
test_loss, test_acc = model.evaluate(test_ds, verbose=0)
print(f"Test accuracy: {test_acc*100:.1f}%")
print(f"Test loss:     {test_loss:.4f}")

In [ ]:
test_loss, test_acc = model.evaluate(test_ds, verbose=1)
print(f"Test accuracy: {test_acc*100:.1f}%")
print(f"Test loss:     {test_loss:.4f}")

### Confusion matrix — where does it get confused?

Each row is the true class, each column is what the model predicted. The diagonal = correct. Off-diagonal cells show which diseases get mixed up.

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

# Collect predictions and true labels from the test set
y_true, y_pred = [], []
for images, labels in test_ds:
    preds = model.predict(images, verbose=0)
    y_pred.extend(np.argmax(preds, axis=1))
    y_true.extend(np.argmax(labels.numpy(), axis=1))
y_true, y_pred = np.array(y_true), np.array(y_pred)

cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(cm, cmap="Greens")
ax.set_xticks(range(5)); ax.set_yticks(range(5))
ax.set_xticklabels([c.replace("_","\n") for c in CLASS_NAMES], fontsize=8)
ax.set_yticklabels([c.replace("_"," ") for c in CLASS_NAMES], fontsize=8)
ax.set_xlabel("Predicted"); ax.set_ylabel("True")
ax.set_title(f"Confusion Matrix (test accuracy {test_acc*100:.1f}%)")
for i in range(5):
    for j in range(5):
        ax.text(j, i, cm[i,j], ha="center", va="center",
                color="white" if cm[i,j] > cm.max()/2 else "black", fontweight="bold")
plt.colorbar(im); plt.tight_layout(); plt.show()

print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, digits=3))

## Step 7 — Predict on individual leaves

The payoff. Here we run the trained model on test images and show its prediction + confidence. Green title = correct, red = wrong. This is exactly what you'd wire up behind a "snap a photo of your plant" app.

In [ ]:
def predict_and_show(dataset, n=8):
    images, labels = next(iter(dataset))
    preds = model.predict(images, verbose=0)
    fig, axes = plt.subplots(2, 4, figsize=(14, 7))
    for i, ax in enumerate(axes.flat):
        if i >= n: break
        ax.imshow(images[i].numpy().astype("uint8"))
        true_idx = np.argmax(labels[i])
        pred_idx = np.argmax(preds[i])
        conf = preds[i][pred_idx] * 100
        correct = (true_idx == pred_idx)
        ax.set_title(
            f"Predicted: {CLASS_NAMES[pred_idx].replace('_',' ')}\n"
            f"({conf:.0f}% sure)   True: {CLASS_NAMES[true_idx].replace('_',' ')}",
            color="green" if correct else "red", fontsize=9)
        ax.axis("off")
    plt.tight_layout(); plt.show()

# Use the unbatched/unshuffled test set for a fair sample
raw_test = tf.keras.utils.image_dataset_from_directory(
    "plant_disease/test", image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=32, label_mode="categorical", shuffle=True, seed=1)
predict_and_show(raw_test, n=8)

### Predict on YOUR OWN image (the fun part for the classroom)

Upload any leaf photo and the model classifies it. On Colab, run this cell and use the upload button.

In [ ]:
# --- Colab: upload your own leaf photo and classify it ---
# from google.colab import files
# uploaded = files.upload()                       # opens a file picker
# fname = list(uploaded.keys())[0]

# For any environment, just point at an image path:
def classify_image(image_path):
    img = tf.keras.utils.load_img(image_path, target_size=(IMG_SIZE, IMG_SIZE))
    arr = tf.keras.utils.img_to_array(img)[None, ...]  # add batch dim
    probs = model.predict(arr, verbose=0)[0]
    plt.imshow(img); plt.axis("off")
    plt.title(f"Prediction: {CLASS_NAMES[np.argmax(probs)].replace('_',' ')} "
              f"({probs.max()*100:.0f}% confident)")
    plt.show()
    # Show all class probabilities
    for i, c in enumerate(CLASS_NAMES):
        print(f"  {c.replace('_',' '):22s} {probs[i]*100:5.1f}%")

# Example:
# classify_image(fname)
print("Ready — call classify_image('your_leaf.jpg') with any leaf photo.")

## 🎓 What the students just built

A complete image classifier, end to end:

1. ✅ Loaded a real high-resolution image dataset
2. ✅ Looked at the data before modelling
3. ✅ Built a tf.data pipeline with automatic resizing
4. ✅ Designed a CNN with the classic conv→pool funnel
5. ✅ Trained it and read the learning curves
6. ✅ Evaluated honestly with a confusion matrix
7. ✅ Made predictions on new images, including their own uploads

### Ideas to extend it (homework)

1. **Bump `IMG_SIZE` to 224** — does accuracy improve? How much slower is training?
2. **Add data augmentation** — insert `layers.RandomFlip()` and `layers.RandomRotation(0.1)` right after the Input. This usually adds a few % accuracy.
3. **Add more classes** — the full PlantVillage dataset has 38 classes across 14 crops.
4. **Try transfer learning** — swap the from-scratch CNN for a pretrained `MobileNetV2` backbone and watch accuracy jump.
5. **Build a web app** — wrap `classify_image()` in a Streamlit or Gradio interface so anyone can drag-drop a photo.

---

*Note on results:* this demo trained a small from-scratch CNN on a balanced 1,100-image subset for 12 epochs. With more data, more epochs, and image augmentation you can push accuracy well into the 90s. The point here is a clean, complete, easy-to-read pipeline — not a leaderboard score.


## Step 8 (Bonus) — Launch a web app, right from this notebook 🚀

The final step in a real project is to **ship it** — turn the model into a web page where anyone can upload a photo and get a prediction. We'll use **Streamlit**, and it all happens in the **one cell below.**

Just run that single cell after training. It **detects where you're running** and does the right thing automatically:
- **On your own computer** (Windows / Mac / Linux): it opens the app in your browser at `http://localhost:8501`
- **On Google Colab**: it installs a tunnel and prints a public link you can open and share

No separate files to manage, no terminal — run the cell and the app comes up.

In [ ]:
# ============================================================
#   ONE CELL: build + launch the Plant Disease web app
#   Works on BOTH your own computer (Windows/Mac/Linux) and Colab.
# ============================================================

# 1) Save the model we just trained so the app can load it
model.save("plant_cnn.keras")

# 2) Write the Streamlit app to a file
app_code = r"""
import streamlit as st
import numpy as np
import tensorflow as tf
from PIL import Image

IMG_SIZE = 128
CLASS_NAMES = ["Apple_Black_Rot", "Apple_Cedar_Rust", "Apple_Healthy",
               "Apple_Scab", "Corn_Common_Rust"]

@st.cache_resource
def load_model():
    return tf.keras.models.load_model("plant_cnn.keras")

model = load_model()

st.title("Plant Disease Detector")
st.write("Upload a photo of a leaf and the model will predict its condition.")

uploaded = st.file_uploader("Choose a leaf image", type=["jpg", "jpeg", "png"])
if uploaded is not None:
    img = Image.open(uploaded).convert("RGB")
    st.image(img, caption="Your image", width=300)

    arr = np.array(img.resize((IMG_SIZE, IMG_SIZE)), dtype="float32")[None, ...]
    probs = model.predict(arr, verbose=0)[0]
    pred = CLASS_NAMES[int(np.argmax(probs))]

    st.subheader("Prediction: " + pred.replace("_", " "))
    st.write("Confidence: {:.1f}%".format(probs.max() * 100))
    st.write("All class probabilities:")
    for c, p in sorted(zip(CLASS_NAMES, probs), key=lambda x: -x[1]):
        st.write("{}: {:.1f}%".format(c.replace("_", " "), p * 100))
        st.progress(float(p))
"""
with open("plant_app.py", "w") as f:
    f.write(app_code)

# 3) Make sure streamlit is installed
import subprocess, sys, time
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "streamlit"], check=False)

# 4) Detect environment: Google Colab or a local machine?
try:
    import google.colab  # noqa
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    # ---- Colab: launch in background + open a public tunnel link ----
    import urllib.request
    subprocess.run(["npm", "install", "-g", "localtunnel"], check=False)
    subprocess.Popen([sys.executable, "-m", "streamlit", "run", "plant_app.py",
                      "--server.port", "8501", "--server.headless", "true"],
                     stdout=open("streamlit_log.txt", "w"), stderr=subprocess.STDOUT)
    time.sleep(8)
    ip = urllib.request.urlopen("https://ipv4.icanhazip.com").read().decode().strip()
    print("=" * 55)
    print("  If the app page asks for a tunnel password, paste:")
    print("  ", ip)
    print("=" * 55)
    print("  Click the https://....loca.lt link below to open your app:")
    get_ipython().system("npx localtunnel --port 8501")
else:
    # ---- Local machine (Windows/Mac/Linux): opens in your browser ----
    print("Starting the app — a browser tab will open at http://localhost:8501")
    print("When you are done, press the stop button (square) to shut it down.")
    get_ipython().system(sys.executable + " -m streamlit run plant_app.py")

In [ ]:
streamlit run plant_app.py